# Amsterdam Airbnb: Price, Regulation & the Tourist City
### Final Individual Project — Data Visualization

**Dataset:** Inside Airbnb — Amsterdam (`listings.csv`, `reviews.csv`, `neighbourhoods.geojson`)
Source: https://insideairbnb.com/amsterdam/ · License: CC BY 4.0 (Murray Cox / Inside Airbnb)

**Story:** Amsterdam caps short-term rentals at 30 nights/year to protect housing stock from
tourism pressure. This notebook investigates whether the data shows evidence of that pressure —
and whether the regulation appears to be shaping host behaviour.

> Place the downloaded CSVs in a `data/` folder next to this notebook before running.


## Setup — run this cell first

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

pd.set_option('display.max_columns', 50)

CVD_SAFE = ['#0072B2', '#E69F00', '#009E73', '#D55E00', '#CC79A7', '#56B4E9']
GREY = '#B0B0B0'
HIGHLIGHT = '#0072B2'

TEMPLATE = go.layout.Template()
TEMPLATE.layout = go.Layout(
    font=dict(family='Helvetica, Arial, sans-serif', size=13, color='#222'),
    plot_bgcolor='white',
    paper_bgcolor='white',
    xaxis=dict(showgrid=False, zeroline=True, zerolinecolor='#888'),
    yaxis=dict(showgrid=False, zeroline=True, zerolinecolor='#888'),
)


In [ ]:
import os
import requests
import pandas as pd

# Download directly into Colab's /content/ folder
BASE = 'https://data.insideairbnb.com/the-netherlands/north-holland/amsterdam/2026-06-15'
files = {
    'listings.csv.gz': f'{BASE}/data/listings.csv.gz',
    'reviews.csv.gz':  f'{BASE}/data/reviews.csv.gz',
    'neighbourhoods.geojson': f'{BASE}/visualisations/neighbourhoods.geojson',
}

for fname, url in files.items():
    path = f'/content/{fname}'
    if not os.path.exists(path):
        print(f"Downloading {fname}...")
        r = requests.get(url, timeout=60)
        r.raise_for_status()
        with open(path, 'wb') as f:
            f.write(r.content)
    else:
        print(f"{fname} already exists, skipping")

print("Download complete.\n")

# Load data (pandas reads .gz directly, no need to unzip)
listings = pd.read_csv("/content/listings.csv.gz")
reviews = pd.read_csv("/content/reviews.csv.gz")

# Clean stray whitespace/quote characters right after loading
for col in ['room_type', 'neighbourhood_cleansed']:
    if col in listings.columns:
        listings[col] = (
            listings[col].astype(str)
            .str.strip()          # remove leading/trailing whitespace
            .str.strip("'\"")     # remove leading/trailing quote characters
        )

print(f"Listings: {listings.shape[0]} rows, {listings.shape[1]} columns")
print(f"Reviews:  {reviews.shape[0]} rows, {reviews.shape[1]} columns")
listings.head(3)

listings.csv.gz already exists, skipping
reviews.csv.gz already exists, skipping
neighbourhoods.geojson already exists, skipping
Download complete.

Listings: 10369 rows, 90 columns
Reviews:  545162 rows, 6 columns


,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,host_url,host_profile_id,host_profile_url,host_name,host_since,hosts_time_as_user_years,hosts_time_as_user_months,hosts_time_as_host_years,hosts_time_as_host_months,host_location,host_about,host_response_time,host_response_rate,host_acceptance_rate,host_is_superhost,...,availability_365,calendar_last_scraped,number_of_reviews,number_of_reviews_ltm,number_of_reviews_l30d,availability_eoy,number_of_reviews_ly,estimated_occupancy_l365d,estimated_revenue_l365d,first_review,last_review,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,28871,https://www.airbnb.com/rooms/28871,20260615212022,2026-06-16,previous scrape,Comfortable double room,Basic bedroom in the center of Amsterdam.,NaN,https://a0.muscache.com/pictures/160889/362340...,124245,https://www.airbnb.com/users/show/124245,1.462510e+18,https://www.airbnb.com/users/profile/146251020...,Edwin,NaN,16.0,1.0,15.0,5.0,"Amsterdam, The Netherlands",Hi,NaN,NaN,NaN,t,...,11,2026-06-16,799,88,5,2,95,255,23970.0,2010-08-22,2026-06-01,4.86,4.89,4.84,4.94,4.94,4.93,4.83,0363 607B EA74 0BD8 2F6F,NaN,2,0,2,0,4.15
1,29051,https://www.airbnb.com/rooms/29051,20260615212022,2026-06-16,previous scrape,Comfortable single / double room,This room can also be rented as a single or a ...,NaN,https://a0.muscache.com/pictures/162009/bd6be2...,124245,https://www.airbnb.com/users/show/124245,1.462510e+18,https://www.airbnb.com/users/profile/146251020...,Edwin,NaN,16.0,1.0,15.0,5.0,"Amsterdam, The Netherlands",Hi,NaN,NaN,NaN,t,...,2,2026-06-16,906,82,6,2,85,255,NaN,2011-03-16,2026-06-01,4.82,4.88,4.83,4.93,4.93,4.88,4.79,0363 607B EA74 0BD8 2F6F,NaN,2,0,2,0,4.88
2,44129,https://www.airbnb.com/rooms/44129,20260615212022,2026-06-24,previous scrape,Luxury design with canal view,"Welcome to my little gem<br /><br />Cozy, brig...",NaN,https://a0.muscache.com/pictures/hosting/Hosti...,187728,https://www.airbnb.com/users/show/187728,1.462512e+18,https://www.airbnb.com/users/profile/146251212...,Tanya,NaN,15.0,10.0,15.0,5.0,"Amsterdam, The Netherlands","I am little bit of a nomad. Born in Belarus, r...",NaN,NaN,NaN,t,...,3,2026-06-24,186,5,0,3,1,39,12233.0,2010-08-16,2026-04-30,4.88,4.81,4.88,4.89,4.89,4.95,4.59,03635399E87602900F47,NaN,4,3,1,0,0.96


In [ ]:
# Clean listings
listings['price'] = (
    listings['price'].astype(str)
    .replace('[\$,]', '', regex=True)
    .astype(float)
)
listings = listings[listings['price'] > 0]

listings['last_review'] = pd.to_datetime(listings.get('last_review'), errors='coerce')
if 'host_is_superhost' in listings.columns:
    listings['host_is_superhost'] = listings['host_is_superhost'].map({'t': True, 'f': False})

reviews['date'] = pd.to_datetime(reviews['date'], errors='coerce')

keep_cols = ['id', 'neighbourhood_cleansed', 'latitude', 'longitude', 'room_type',
             'price', 'minimum_nights', 'number_of_reviews', 'review_scores_rating',
             'availability_365', 'host_id', 'calculated_host_listings_count',
             'host_is_superhost', 'last_review']
keep_cols = [c for c in keep_cols if c in listings.columns]
listings = listings.dropna(subset=['price', 'neighbourhood_cleansed'])[keep_cols]

print(f"After cleaning: {listings.shape[0]} listings")
listings.head(3)


After cleaning: 6377 listings


<>:4: SyntaxWarning: invalid escape sequence '\$'
<>:4: SyntaxWarning: invalid escape sequence '\$'
/tmp/ipykernel_4515/976823752.py:4: SyntaxWarning: invalid escape sequence '\$'
  .replace('[\$,]', '', regex=True)


,id,neighbourhood_cleansed,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,review_scores_rating,availability_365,host_id,calculated_host_listings_count,host_is_superhost,last_review
0,28871,Centrum-West,52.36775,4.89092,Private room,94.00,1.0,799,4.86,11,124245,2,True,2026-06-01
2,44129,Centrum-West,52.38211,4.88630,Entire home/apt,313.67,3.0,186,4.88,3,187728,4,True,2026-04-30
5,49552,Centrum-West,52.38028,4.89089,Entire home/apt,440.27,2.0,656,4.94,278,225987,1,True,2026-06-07


## Preliminary EDA
*(exploration only — does not count toward the 10 analytical questions)*


In [ ]:
print(listings['neighbourhood_cleansed'].value_counts())
print()
print(listings['room_type'].value_counts())
print()
print(listings['price'].describe())


neighbourhood_cleansed
De Baarsjes - Oud-West                    1066
Centrum-West                               851
De Pijp - Rivierenbuurt                    696
Centrum-Oost                               666
Zuid                                       435
Westerpark                                 402
Oud-Oost                                   383
Oud-Noord                                  278
Bos en Lommer                              257
Oostelijk Havengebied - Indische Buurt     231
Noord-West                                 205
Watergraafsmeer                            178
Noord-Oost                                 124
Slotervaart                                118
IJburg - Zeeburgereiland                   116
Geuzenveld - Slotermeer                     94
Buitenveldert - Zuidas                      87
Bijlmer-Centrum                             47
De Aker - Nieuw Sloten                      46
Osdorp                                      40
Gaasperdam - Driemond                

---
## Question 1 — Which neighbourhoods command the biggest price premium?


In [ ]:
region_avg = (listings.groupby('neighbourhood_cleansed')['price']
              .mean()
              .reset_index()
              .sort_values('price'))

top_hood = region_avg.iloc[-1]['neighbourhood_cleansed']
colors = [HIGHLIGHT if n == top_hood else GREY for n in region_avg['neighbourhood_cleansed']]

fig1 = go.Figure(go.Bar(
    x=region_avg['price'], y=region_avg['neighbourhood_cleansed'],
    orientation='h', marker_color=colors,
    text=region_avg['price'].round(0).astype(int), textposition='outside'
))
fig1.update_layout(
    template=TEMPLATE, height=600,
    title=f"{top_hood} commands the steepest Airbnb price premium in Amsterdam",
    xaxis_title='Average price per night (EUR)', yaxis_title=None,
    margin=dict(l=180)
)
fig1.show()


**Takeaway:** *(write 2–3 sentences once you see the real result — is the premium explained by canal-ring proximity, listing quality, or something else?)*

---
## Question 2 — Does hosting activity cluster around the 30-night regulatory cap?
Amsterdam limits short-term rentals to 30 nights/year (occupied ≥ 335 of 365 available nights implies compliance at the edge).


In [ ]:
fig2 = px.histogram(
    listings, x='availability_365', nbins=50,
    color_discrete_sequence=[GREY]
)
fig2.add_vline(x=335, line_dash='dash', line_color=HIGHLIGHT,
               annotation_text='365 - 30 night cap', annotation_position='top')
fig2.update_layout(
    template=TEMPLATE,
    title="Availability patterns show whether hosts cluster around the 30-night rental cap",
    xaxis_title='Days available per year', yaxis_title='Number of listings'
)
fig2.show()


**Takeaway:** *(does a spike appear near the cap line, suggesting hosts are optimizing around the regulation?)*

---
## Question 3 — How does the entire-home vs. private-room mix vary by neighbourhood?


In [ ]:
print(listings['room_type'].unique())

['Private room' 'Entire home/apt' 'Hotel room' 'Shared room']


In [ ]:
print([repr(x) for x in listings['room_type'].unique()])

["'Private room'", "'Entire home/apt'", "'Hotel room'", "'Shared room'"]


In [ ]:
mix = (listings.groupby(['neighbourhood_cleansed', 'room_type'], observed=True).size()
       .reset_index(name='count'))

mix['neighbourhood_cleansed'] = mix['neighbourhood_cleansed'].astype(str)
mix['room_type'] = mix['room_type'].astype(str)

totals = mix.groupby('neighbourhood_cleansed')['count'].transform('sum')
mix['share'] = mix['count'] / totals

main_type = str(listings['room_type'].astype(str).value_counts().idxmax())
print(f"Sorting by share of: {main_type}")

order_df = mix[mix['room_type'] == main_type].sort_values('share', ascending=True)
order = order_df['neighbourhood_cleansed'].tolist()
remaining = [n for n in mix['neighbourhood_cleansed'].unique() if n not in order]
order = remaining + order

# No category_orders here — set the axis order afterwards instead
fig3 = px.bar(
    mix, x='share', y='neighbourhood_cleansed', color='room_type',
    orientation='h',
    color_discrete_sequence=CVD_SAFE
)
fig3.update_yaxes(categoryorder='array', categoryarray=order)
fig3.update_layout(
    template=TEMPLATE, height=600, barmode='stack',
    title=f"{main_type} listings dominate central neighbourhoods, edging out long-term housing stock",
    xaxis_title='Share of listings', yaxis_title=None, xaxis_tickformat='.0%',
    margin=dict(l=180)
)
fig3.show()

Sorting by share of: Entire home/apt


---
## Question 4 — Do professional (multi-listing) hosts price differently?


In [ ]:
listings['host_type'] = np.where(
    listings['calculated_host_listings_count'] > 1, 'Multi-listing host', 'Single-listing host'
)
host_price = listings.groupby('host_type')['price'].mean().reset_index()

fig4 = go.Figure(go.Bar(
    x=host_price['host_type'], y=host_price['price'],
    marker_color=[HIGHLIGHT, GREY],
    text=host_price['price'].round(0).astype(int), textposition='outside'
))
fig4.update_layout(
    template=TEMPLATE,
    title="Multi-listing hosts price their Amsterdam Airbnbs higher than single-listing hosts",
    yaxis_title='Average price per night (EUR)', xaxis_title=None
)
fig4.show()


---
## Question 5 — How does review volume (a proxy for bookings) trend over time?


In [ ]:
monthly = (reviews.set_index('date').resample('ME').size().reset_index(name='reviews'))

fig5 = px.line(monthly, x='date', y='reviews', color_discrete_sequence=[HIGHLIGHT])
fig5.update_layout(
    template=TEMPLATE,
    title="Review activity reveals Amsterdam's tourist season peaks in spring and summer",
    xaxis_title=None, yaxis_title='Reviews per month'
)
fig5.show()


---
## Question 6 — Do minimum-night requirements differ between central and outer neighbourhoods?


In [ ]:
min_nights = (listings.groupby('neighbourhood_cleansed')['minimum_nights']
              .median().reset_index().sort_values('minimum_nights'))

fig6 = go.Figure(go.Bar(
    x=min_nights['minimum_nights'], y=min_nights['neighbourhood_cleansed'],
    orientation='h', marker_color=GREY
))
fig6.update_layout(
    template=TEMPLATE, height=600,
    title="Minimum-stay rules vary sharply across Amsterdam's neighbourhoods",
    xaxis_title='Median minimum nights', yaxis_title=None, margin=dict(l=180)
)
fig6.show()


---
## Question 7 — Do cheaper listings score worse, or is that a myth?


In [ ]:
bins = [0, 100, 150, 200, 300, listings['price'].max()]
labels = ['<100', '100-150', '150-200', '200-300', '300+']
listings['price_band'] = pd.cut(listings['price'], bins=bins, labels=labels)

score_by_band = (listings.dropna(subset=['review_scores_rating'])
                  .groupby('price_band')['review_scores_rating'].mean().reset_index())

fig7 = go.Figure(go.Bar(
    x=score_by_band['price_band'], y=score_by_band['review_scores_rating'],
    marker_color=GREY
))
fig7.update_layout(
    template=TEMPLATE,
    title="Review scores stay flat across price bands — price is not a quality signal",
    xaxis_title='Price band (EUR)', yaxis_title='Average review score'
)
fig7.show()


/tmp/ipykernel_4515/717135847.py:6: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



---
## Question 8 — Where in the city are prices highest?


In [ ]:
fig8 = px.scatter_mapbox(
    listings, lat='latitude', lon='longitude', color='price',
    color_continuous_scale='Blues', size_max=8, zoom=11,
    mapbox_style='carto-positron',
    range_color=(listings['price'].quantile(0.05), listings['price'].quantile(0.95))
)
fig8.update_layout(
    template=TEMPLATE,
    title="A second price cluster emerges outside the historic centre",
    margin=dict(l=0, r=0, t=40, b=0)
)
fig8.show()


---
## Question 9 — Does Superhost status correlate with price or with occupancy?


In [ ]:
if 'host_is_superhost' in listings.columns:
    comp = listings.groupby('host_is_superhost').agg(
        avg_price=('price', 'mean'),
        avg_reviews=('number_of_reviews', 'mean')
    ).reset_index()
    comp['host_is_superhost'] = comp['host_is_superhost'].map({True: 'Superhost', False: 'Regular host'})

    fig9 = go.Figure()
    fig9.add_bar(x=comp['host_is_superhost'], y=comp['avg_price'], name='Avg price (EUR)',
                 marker_color=HIGHLIGHT)
    fig9.update_layout(
        template=TEMPLATE,
        title="Superhost status tracks with review volume more than with price premium",
        yaxis_title='Average price (EUR)'
    )
    fig9.show()
else:
    print("host_is_superhost column not found — check your listings.csv version")


---
## Question 10 — Where (if anywhere) does the shared-room type still exist?


In [ ]:
shared = (listings[listings['room_type'] == 'Shared room']
          .groupby('neighbourhood_cleansed').size().reset_index(name='count')
          .sort_values('count'))

fig10 = go.Figure(go.Bar(
    x=shared['count'], y=shared['neighbourhood_cleansed'],
    orientation='h', marker_color=GREY
))
fig10.update_layout(
    template=TEMPLATE, height=500,
    title="Shared-room listings have nearly disappeared from Amsterdam's Airbnb market",
    xaxis_title='Number of listings', yaxis_title=None, margin=dict(l=180)
)
fig10.show()
